# Project 51 — Local Web Research Agent

**Search the web, compare sources, and write cited answers — all locally with Ollama.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Search | DuckDuckGo (live) + simulated fallback |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to define **custom tools** in LangChain and wire them to a local LLM
2. How to implement a **search → summarize → cite** research pipeline
3. How to compare information from **multiple sources** and produce a grounded answer
4. How to build a **fallback-safe** agent that works both online and offline
5. How to evaluate agent output for **groundedness** and citation quality

## 2 · Why This Project Matters

Most LLM answers lack citations and can hallucinate. A research agent that actually
**retrieves** information before answering produces more trustworthy output. By running
everything locally, you keep your queries private and learn how tool-calling works under
the hood — a skill needed for every production agent system.

## 3 · Environment Setup

**Prerequisites:**
- Ollama installed and running (`ollama serve`)
- Model pulled: `ollama pull qwen3:8b`
- Python packages below

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community duckduckgo-search

## 4 · Imports and Configuration

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "qwen3:8b"
TEMPERATURE = 0.2

llm = ChatOllama(model=MODEL, temperature=TEMPERATURE)
print(f"LLM ready: {MODEL}")

## 5 · Define the Search Tool

We try DuckDuckGo first (works without API keys). If that fails (no internet,
rate-limited), we fall back to a built-in knowledge base so the notebook remains
runnable offline.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated knowledge base (offline fallback)
# ---------------------------------------------------------------------------
KNOWLEDGE_BASE = {
    "RAG vs fine-tuning": [
        {"source": "Hugging Face Blog, 2024",
         "text": "RAG retrieves external documents at inference time, keeping the model frozen. "
                 "Fine-tuning updates model weights on new data. RAG is cheaper to maintain "
                 "because you only update the document store, not the model."},
        {"source": "Anthropic Research, 2024",
         "text": "Fine-tuning is best for style/tone adaptation and domain-specific vocabulary. "
                 "RAG excels when you need factual accuracy with traceable citations."},
        {"source": "Google DeepMind Report, 2025",
         "text": "Hybrid approaches (RAG + lightweight fine-tuning) outperform either alone "
                 "on knowledge-intensive benchmarks by 12-18%."},
    ],
    "local LLM inference": [
        {"source": "Ollama Docs, 2025",
         "text": "Ollama supports Llama 3, Mistral, Qwen 3, Phi-4, Gemma 2 and 100+ models "
                 "for fully local execution with GPU acceleration."},
        {"source": "MLOps Weekly, 2025",
         "text": "Local inference eliminates API costs and latency. A single RTX 4090 can "
                 "serve 8B models at 60+ tokens/s with 4-bit quantization."},
    ],
    "vector databases comparison": [
        {"source": "DB-Engines, 2025",
         "text": "ChromaDB is popular for prototyping; Qdrant and Weaviate lead in production. "
                 "FAISS remains fastest for pure ANN search on a single machine."},
        {"source": "LlamaIndex Benchmark, 2025",
         "text": "For local RAG under 1M documents, ChromaDB and FAISS offer the best "
                 "latency/accuracy tradeoff without any external infrastructure."},
    ],
}


def _search_offline(query: str) -> list:
    """Match query against the local knowledge base."""
    query_lower = query.lower()
    results = []
    for topic, entries in KNOWLEDGE_BASE.items():
        if any(word in query_lower for word in topic.lower().split()):
            results.extend(entries)
    if not results:
        results = [{"source": "General Knowledge",
                    "text": "No specific offline results. Try refining your query."}]
    return results


def _search_online(query: str, max_results: int = 4) -> list:
    """Try DuckDuckGo; return empty list on failure."""
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(query, max_results=max_results))
        return [{"source": r.get("href", "web"), "text": r.get("body", "")} for r in raw]
    except Exception as exc:
        print(f"  [online search unavailable: {exc}]")
        return []


@tool
def web_search(query: str) -> str:
    """Search the web for information. Returns structured results with sources."""
    results = _search_online(query)
    if not results:
        results = _search_offline(query)
    formatted = []
    for i, r in enumerate(results, 1):
        formatted.append(f"[Source {i}] {r['source']}\n{r['text']}")
    return "\n\n".join(formatted)


print("web_search tool defined")
print("Quick test:")
print(web_search.invoke("RAG vs fine-tuning"))

## 6 · Define the Summarization and Comparison Tools

These tools take a block of text (typically the search results) and produce concise
outputs via the local LLM.

In [ ]:
@tool
def summarize_sources(text: str) -> str:
    """Summarize retrieved text into structured key points."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research summarizer. Produce 3-5 bullet points that capture "
         "the most important facts. Keep each bullet under 30 words. "
         "Preserve source attributions."),
        ("human", "{text}"),
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": text})


@tool
def compare_sources(text: str) -> str:
    """Compare viewpoints across sources and note agreements or conflicts."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research analyst. Given multiple source extracts, identify:\n"
         "1. Points of AGREEMENT across sources\n"
         "2. Points of DISAGREEMENT or nuance\n"
         "3. Information gaps\n"
         "Be concise."),
        ("human", "{text}"),
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": text})


tools = [web_search, summarize_sources, compare_sources]
print(f"Tools defined: {[t.name for t in tools]}")

## 7 · Build the Research Pipeline

Rather than relying on fully autonomous tool-calling (which can be unreliable with
small local models), we build a **structured pipeline**:

```
Question → Search → Summarize → Compare → Write Cited Answer
```

This is a common practical pattern: use the LLM for language tasks (summarizing,
comparing, writing) while keeping the **control flow deterministic**.

In [ ]:
def research(question: str, verbose: bool = True) -> dict:
    """Run the full research pipeline on a question."""
    if verbose:
        print(f"\n{'='*70}")
        print(f"RESEARCH QUESTION: {question}")
        print(f"{'='*70}")

    # Step 1 -- Search
    if verbose:
        print("\n[Step 1] Searching...")
    raw_sources = web_search.invoke(question)
    if verbose:
        preview = raw_sources[:300] + "..." if len(raw_sources) > 300 else raw_sources
        print(textwrap.indent(preview, "  "))

    # Step 2 -- Summarize
    if verbose:
        print("\n[Step 2] Summarizing...")
    summary = summarize_sources.invoke(raw_sources)
    if verbose:
        print(textwrap.indent(summary, "  "))

    # Step 3 -- Compare sources
    if verbose:
        print("\n[Step 3] Comparing sources...")
    comparison = compare_sources.invoke(raw_sources)
    if verbose:
        print(textwrap.indent(comparison, "  "))

    # Step 4 -- Write cited answer
    if verbose:
        print("\n[Step 4] Writing cited answer...")

    answer_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research writer. Write a clear, well-structured answer to the "
         "question using ONLY the provided research. Rules:\n"
         "- Cite sources inline using [Source N] notation\n"
         "- If sources disagree, note the disagreement\n"
         "- End with a 'Sources' list\n"
         "- Do NOT add information beyond what the research provides"),
        ("human",
         "Question: {question}\n\n"
         "Raw Sources:\n{sources}\n\n"
         "Summary:\n{summary}\n\n"
         "Source Comparison:\n{comparison}"),
    ])
    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({
        "question": question,
        "sources": raw_sources,
        "summary": summary,
        "comparison": comparison,
    })

    if verbose:
        print("\n" + "-"*70)
        print("FINAL ANSWER:")
        print("-"*70)
        print(answer)

    return {
        "question": question,
        "sources": raw_sources,
        "summary": summary,
        "comparison": comparison,
        "answer": answer,
        "timestamp": datetime.now().isoformat(),
    }


print("Research pipeline ready.")

## 8 · Run Research Queries

Let's run the agent on three different questions to see how it handles different
topics and source combinations.

In [ ]:
result_1 = research("What is the difference between RAG and fine-tuning for LLMs?")

In [ ]:
result_2 = research("What are the best options for local LLM inference in 2025?")

In [ ]:
result_3 = research("How do vector databases compare for local RAG applications?")

## 9 · Evaluate Output Quality

A good research agent should produce **grounded** answers. Let's check whether the
answers actually cite their sources and don't introduce unsupported claims.

In [ ]:
def evaluate_answer(result: dict) -> dict:
    """Simple automated evaluation of a research result."""
    answer = result["answer"]
    sources = result["sources"]

    # Check for citations
    citation_count = answer.count("[Source")

    # Check answer length
    word_count = len(answer.split())
    length_ok = 50 <= word_count <= 500

    # Ask LLM to judge groundedness
    judge_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a fact-checker. Given sources and an answer, respond with ONLY "
         "a JSON object:\n"
         '{"grounded": true/false, "unsupported_claims": ["..."], "score": 1-5}'),
        ("human", "Sources:\n{sources}\n\nAnswer:\n{answer}"),
    ])
    judge_chain = judge_prompt | llm | StrOutputParser()
    judgment_raw = judge_chain.invoke({"sources": sources, "answer": answer})

    return {
        "question": result["question"],
        "citation_count": citation_count,
        "word_count": word_count,
        "length_ok": length_ok,
        "llm_judgment": judgment_raw,
    }


print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
for res in [result_1, result_2, result_3]:
    ev = evaluate_answer(res)
    print(f"\nQ: {ev['question'][:60]}...")
    print(f"  Citations found: {ev['citation_count']}")
    print(f"  Word count:      {ev['word_count']} ({'ok' if ev['length_ok'] else 'check'})")
    print(f"  LLM judgment:    {ev['llm_judgment'][:200]}")

## 10 · Save Research Results

In [ ]:
all_results = [result_1, result_2, result_3]
with open("research_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"Saved {len(all_results)} research results to research_results.json")

## 11 · Limitations

- **Offline fallback** uses a small hardcoded knowledge base — real search gives broader coverage
- **Small local models** sometimes produce weak citations or hallucinate details
- **No deduplication** of near-identical sources
- **Structured pipeline** means the agent can't decide to search again if results are poor
- DuckDuckGo has rate limits; for production use consider a local search index

## 12 · How to Improve This Project

1. **Add a re-search loop**: if the LLM detects insufficient evidence, trigger another search
2. **Use LangGraph** to make the pipeline stateful with branching and human-in-the-loop
3. **Add a local reranker** (e.g., a cross-encoder) to filter the best search results
4. **Persist search results** in a vector store for future queries
5. **Switch to PydanticAI** for structured, type-safe tool calling

## 13 · Mini Challenge

1. Add a fourth topic to `KNOWLEDGE_BASE` and run a query against it
2. Modify the `compare_sources` tool to output a structured Markdown table
3. Add a `follow_up_question` tool that generates a follow-up question from the answer

## 14 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| Tool definition | `@tool` decorator with docstrings for LLM understanding |
| Structured pipeline | Search → Summarize → Compare → Cite |
| Fallback design | Online search with offline graceful degradation |
| Output evaluation | Automated citation counting + LLM-as-judge |
| Local-first | Everything runs on Ollama — no cloud API needed |